In [1]:
import scanpy as sc
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import scipy.sparse as sp
import numpy as np
from os.path import join
import os
from matplotlib.colors import ListedColormap
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import re
import logging
from typing import Optional, Sequence, Union, Tuple
from scipy import sparse
import glob
from matplotlib.ticker import FuncFormatter
from typing import Tuple  # ✅ Python 3.8 compatible

In [2]:
combined_data_annotated = sc.read_h5ad('/local/hailey/PLX/all_samples_bin100_process_anno.h5ad')#process data
meta = pd.read_csv('/local/hailey/PLX/all_samples_bin100_process_meta.csv', index_col=0)
combined_data_annotated.obs = meta  #domain annotation --- 'domains_c' column

In [3]:
combined_data = sc.read_h5ad("/local/hailey/PLX/all_samples_bin100.h5ad")#raw data
combined_data

AnnData object with n_obs × n_vars = 367820 × 49340
    obs: 'nCount_Spatial', 'nFeature_Spatial', 'percent.mito', 'orig.ident', 'x', 'y', 'Sample', 'ChipID', 'Tissue', 'MouseID', 'Group', 'Sex', 'SlicingDate', 'batch'
    var: 'ENSid-0', 'n_cells-0', 'n_counts-0', 'mean_umi-0', 'ENSid-1', 'n_cells-1', 'n_counts-1', 'mean_umi-1', 'ENSid-10', 'n_cells-10', 'n_counts-10', 'mean_umi-10', 'ENSid-11', 'n_cells-11', 'n_counts-11', 'mean_umi-11', 'ENSid-12', 'n_cells-12', 'n_counts-12', 'mean_umi-12', 'ENSid-13', 'n_cells-13', 'n_counts-13', 'mean_umi-13', 'ENSid-14', 'n_cells-14', 'n_counts-14', 'mean_umi-14', 'ENSid-15', 'n_cells-15', 'n_counts-15', 'mean_umi-15', 'ENSid-16', 'n_cells-16', 'n_counts-16', 'mean_umi-16', 'ENSid-17', 'n_cells-17', 'n_counts-17', 'mean_umi-17', 'ENSid-18', 'n_cells-18', 'n_counts-18', 'mean_umi-18', 'ENSid-19', 'n_cells-19', 'n_counts-19', 'mean_umi-19', 'ENSid-2', 'n_cells-2', 'n_counts-2', 'mean_umi-2', 'ENSid-3', 'n_cells-3', 'n_counts-3', 'mean_umi-3', 'ENS

In [4]:
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Helvetica"]  
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

In [5]:
# ========== 小工具：安全文件名 ==========
def _safe_name(x: str, maxlen: int = 120) -> str:
    x = str(x)
    x = re.sub(r"[^\w\-_\.]+", "_", x)  # 空格/斜杠等替换为 _
    return x[:maxlen]


# ========== 核心：pseudobulk ==========
def create_pseudobulk(
    adata: sc.AnnData,
    groupby_cols: Union[str, Sequence[str]],
    celltype_col: Optional[str] = None,
    celltype_filter: Optional[Union[str, Sequence[str]]] = None,
    layer: Optional[str] = "counts",
    use_raw: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    按 group 将 counts 求和生成 pseudobulk:
      - pseudobulk_df: rows=伪bulk样本, cols=genes
      - pseudobulk_meta_df: 每个伪bulk的meta（group列 + n_cells 等）
    """
    if isinstance(groupby_cols, str):
        groupby_cols = [groupby_cols]
    groupby_cols = list(groupby_cols)

    # 检查 groupby 列
    missing = [c for c in groupby_cols if c not in adata.obs.columns]
    if missing:
        raise ValueError(f"groupby_cols 不在 adata.obs 中: {missing}")

    # 可选：按 celltype 过滤
    if celltype_col is not None and celltype_filter is not None:
        if celltype_col not in adata.obs.columns:
            raise ValueError(f"celltype_col '{celltype_col}' 不在 adata.obs 中")

        if isinstance(celltype_filter, str):
            celltype_filter = [celltype_filter]
        else:
            celltype_filter = list(celltype_filter)

        mask = adata.obs[celltype_col].isin(celltype_filter)
        if mask.sum() == 0:
            raise ValueError(f"没有找到 celltype_filter={celltype_filter} 对应的细胞")
        ad = adata[mask].copy()
    else:
        ad = adata
        celltype_filter = None  # 统一

    # 选择 counts 矩阵
    if layer is not None and layer in ad.layers:
        M = ad.layers[layer]
        genes = ad.var_names
    elif use_raw and ad.raw is not None:
        M = ad.raw.X
        genes = ad.raw.var_names
    else:
        M = ad.X
        genes = ad.var_names

    # 让稀疏矩阵切片更快
    if sparse.issparse(M) and not sparse.isspmatrix_csr(M):
        M = M.tocsr()

    meta = ad.obs[groupby_cols].copy()
    meta["_row"] = np.arange(ad.n_obs)

    grouped = meta.groupby(groupby_cols, dropna=False)

    pb_counts = []
    pb_meta = []

    for key, g in grouped:
        rows = g["_row"].to_numpy()
        summed = M[rows].sum(axis=0)
        summed = np.asarray(summed).ravel()

        pb_counts.append(summed)

        key_tuple = key if isinstance(key, tuple) else (key,)
        md = dict(zip(groupby_cols, key_tuple))
        md["n_cells"] = len(rows)

        # 只有真的做了过滤才写 celltype
        if celltype_col is not None and celltype_filter is not None:
            md["celltype"] = celltype_filter[0] if len(celltype_filter) == 1 else celltype_filter

        pb_meta.append(md)

    pseudobulk_df = pd.DataFrame(np.vstack(pb_counts), columns=genes)
    pseudobulk_meta_df = pd.DataFrame(pb_meta)

    # 给 counts 一个可读 index
    pseudobulk_df.index = pseudobulk_meta_df[groupby_cols].astype(str).agg("|".join, axis=1).values

    return pseudobulk_df, pseudobulk_meta_df



In [6]:
def run_region_deg_deseq2_by_sex(
    adata,
    sex_col="Sex",                 
    region_col="domains_c",
    sample_col="Sample",
    condition_col="Group",        
    layer="counts",
    use_raw=False,
    min_total_counts_per_gene=10,
    out_dir="DEG_by_region_by_sex",
    padj_cutoff=0.05,
    lfc_cutoff=1.0,
    sexes=None,                   
    celltype_col=None,
    celltype_filter=None,
):
    if sex_col not in adata.obs.columns:
        raise ValueError(f"sex_col '{sex_col}' 不在 adata.obs 中")

    os.makedirs(out_dir, exist_ok=True)

    if sexes is None:
        sexes = adata.obs[sex_col].dropna().unique().tolist()

    all_summary = []

    for sex in sexes:
        ad_s = adata[adata.obs[sex_col] == sex].copy()
        if ad_s.n_obs == 0:
            continue

        sex_dir = os.path.join(out_dir, f"sex_{_safe_name(sex)}")
        os.makedirs(sex_dir, exist_ok=True)

        regions = ad_s.obs[region_col].dropna().unique().tolist()
        summary = []

        for region in regions:
            try:
                ad_r = ad_s[ad_s.obs[region_col] == region].copy()

                # ===== pseudobulk：按 sample + condition 聚合（sex 已经分开，所以不再放 sex_col）=====
                groupby_cols = [sample_col, condition_col]

                pb_counts, pb_meta = create_pseudobulk(
                    ad_r,
                    groupby_cols=groupby_cols,
                    celltype_col=celltype_col,
                    celltype_filter=celltype_filter,
                    layer=layer,
                    use_raw=use_raw,
                )

                pb_counts = pb_counts.round().astype(int)

                # 过滤低表达基因
                keep_gene = (pb_counts.sum(axis=0) >= min_total_counts_per_gene)
                pb_counts = pb_counts.loc[:, keep_gene]

                # 每组至少 2 个 sample 才跑
                n_rep = pb_meta.groupby(condition_col)[sample_col].nunique()
                if ("PLX" not in n_rep.index) or ("Control" not in n_rep.index) or (n_rep["PLX"] < 2) or (n_rep["Control"] < 2):
                    summary.append({
                        "sex": sex,
                        "region": region,
                        "status": "skipped_insufficient_replicates",
                        "n_rep_PLX": int(n_rep.get("PLX", 0)),
                        "n_rep_Control": int(n_rep.get("Control", 0)),
                    })
                    continue

                # 对齐
                pb_meta = pb_meta.reset_index(drop=True)
                pb_counts.index = pb_meta.index

                # ===== DESeq2：设计只含 condition（因为 sex 已经分层）=====
                dds = DeseqDataSet(
                    counts=pb_counts,
                    metadata=pb_meta,
                    design_factors=[condition_col],
                )
                dds.deseq2()

                stat = DeseqStats(
                    dds,
                    contrast=(condition_col, "PLX", "Control"),
                    alpha=padj_cutoff,
                )
                stat.summary()
                res = stat.results_df.sort_values("padj")

                # 保存
                region_safe = _safe_name(region)
                out_all = os.path.join(sex_dir, f"DESeq2_all_{region_safe}.csv")
                out_sig = os.path.join(sex_dir, f"DEG_sig_{region_safe}.csv")

                res.to_csv(out_all)

                deg_sig = res.dropna(subset=["padj", "log2FoldChange"])
                deg_sig = deg_sig[(deg_sig["padj"] < padj_cutoff) & (deg_sig["log2FoldChange"].abs() >= lfc_cutoff)]
                deg_sig.to_csv(out_sig)

                summary.append({
                    "sex": sex,
                    "region": region,
                    "status": "ok",
                    "n_bulk": int(pb_counts.shape[0]),
                    "n_genes": int(pb_counts.shape[1]),
                    "n_deg_sig": int(deg_sig.shape[0]),
                    "out_all": out_all,
                    "out_sig": out_sig,
                })

            except Exception as e:
                summary.append({"sex": sex, "region": region, "status": "failed", "error": str(e)})

        summary_df = pd.DataFrame(summary)
        summary_df.to_csv(os.path.join(sex_dir, "summary.csv"), index=False)
        all_summary.append(summary_df)

    all_summary_df = pd.concat(all_summary, ignore_index=True) if len(all_summary) > 0 else pd.DataFrame()
    all_summary_df.to_csv(os.path.join(out_dir, "summary_all_sexes.csv"), index=False)
    return all_summary_df

In [7]:
combined_data.obs = combined_data_annotated.obs
summary_all = run_region_deg_deseq2_by_sex(
    combined_data,
    sex_col="Sex",          # 改成你实际列名
    region_col="domains_c",
    sample_col="Sample",
    condition_col="Group",  # 必须包含 "PLX"/"Control"
    layer="counts",
    out_dir="DEG_by_region_by_sex",
)

print(summary_all.head())


/home/hailey/conda/envs/st/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 2.26 seconds.

Fitting dispersion trend curve...
... done in 0.33 seconds.

Fitting MAP dispersions...
... done in 2.32 seconds.

Fitting LFCs...
... done in 2.47 seconds.

Refitting 0 outliers.

Running Wald tests...
... done in 2.36 seconds.



Log2 fold change & Wald test p-value: Group PLX vs Control
                baseMean  log2FoldChange     lfcSE      stat    pvalue  \
real_gene_name                                                           
0610012G03Rik   1.107066       -0.008205  3.073627 -0.002669  0.997870   
1110008P14Rik   1.627640       -1.098219  2.701998 -0.406447  0.684414   
1110038F14Rik   1.967634       -0.588418  3.017976 -0.194971  0.845416   
1110051M20Rik   2.699002        2.443856  1.963225  1.244818  0.213199   
1190007I07Rik   1.120282        0.522212  3.057334  0.170806  0.864376   
...                  ...             ...       ...       ...       ...   
mt-Tr           1.678557        4.073796  2.949167  1.381338  0.167175   
mt-Ts2          6.273068        1.338984  1.179041  1.135656  0.256101   
mt-Tt           3.349597        1.437660  1.676347  0.857615  0.391105   
mt-Tv           7.654294       -0.863167  1.361705 -0.633887  0.526155   
n-R5-8s1        0.881119       -1.010369  3.098613 -0

/home/hailey/conda/envs/st/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
Fitting size factors...
... done in 0.02 seconds.

Fitting dispersions...
... done in 12.06 seconds.

Fitting dispersion trend curve...
... done in 3.22 seconds.

Fitting MAP dispersions...
... done in 10.99 seconds.

Fitting LFCs...
... done in 7.44 seconds.

Refitting 0 outliers.

Running Wald tests...
... done in 6.64 seconds.



Log2 fold change & Wald test p-value: Group PLX vs Control
                  baseMean  log2FoldChange     lfcSE      stat    pvalue  \
real_gene_name                                                             
0610005C13Rik    38.732994       -0.026761  0.195359 -0.136986  0.891042   
0610006L08Rik     1.146082        0.745287  1.120365  0.665218  0.505911   
0610009B22Rik   769.379456        0.150386  0.189517  0.793523  0.427473   
0610009E02Rik    78.000565       -0.104087  0.108009 -0.963686  0.335203   
0610009L18Rik   168.137283        0.135301  0.117515  1.151353  0.249587   
...                    ...             ...       ...       ...       ...   
n-R5s71           1.262016        1.752259  1.275712  1.373553  0.169580   
n-R5s77           2.444651       -0.042240  0.858856 -0.049182  0.960774   
n-R5s88           1.539614        0.025371  0.929877  0.027284  0.978233   
n-R5s95           1.389311        1.350958  1.171872  1.152820  0.248984   
n-R5s96           1.365399   

/home/hailey/conda/envs/st/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
Fitting size factors...
... done in 0.02 seconds.

Fitting dispersions...
... done in 12.35 seconds.

Fitting dispersion trend curve...
... done in 3.12 seconds.

Fitting MAP dispersions...
... done in 10.77 seconds.

Fitting LFCs...
... done in 7.14 seconds.

Refitting 0 outliers.

Running Wald tests...
... done in 6.24 seconds.



Log2 fold change & Wald test p-value: Group PLX vs Control
                  baseMean  log2FoldChange     lfcSE      stat    pvalue  \
real_gene_name                                                             
0610005C13Rik    27.742182       -0.385521  0.171600 -2.246626  0.024664   
0610006L08Rik     1.253329        0.136215  0.946452  0.143922  0.885562   
0610009B22Rik   663.503967        0.196922  0.171082  1.151039  0.249716   
0610009E02Rik    49.220673        0.123256  0.149831  0.822631  0.410718   
0610009L18Rik   115.058067        0.029900  0.140169  0.213312  0.831084   
...                    ...             ...       ...       ...       ...   
n-R5s33           2.038712       -0.047376  0.893916 -0.052999  0.957733   
n-R5s58           0.829150       -1.847814  1.256945 -1.470084  0.141539   
n-R5s71           0.973386        0.666099  1.192197  0.558715  0.576356   
n-R5s77           1.257307        1.136418  1.095881  1.036990  0.299740   
n-R5s95           1.300431   

... (42 outputs left out)

/home/hailey/conda/envs/st/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
Fitting size factors...
... done in 0.01 seconds.

Fitting dispersions...
... done in 10.00 seconds.

Fitting dispersion trend curve...
... done in 1.88 seconds.

Fitting MAP dispersions...
... done in 8.22 seconds.

Fitting LFCs...
... done in 5.69 seconds.

Refitting 0 outliers.

Running Wald tests...
... done in 4.95 seconds.



Log2 fold change & Wald test p-value: Group PLX vs Control
                  baseMean  log2FoldChange     lfcSE      stat    pvalue  \
real_gene_name                                                             
0610005C13Rik     2.482727        0.595985  0.656098  0.908377  0.363679   
0610009B22Rik   100.987656        0.013958  0.106169  0.131472  0.895402   
0610009E02Rik     5.589934       -0.447309  0.386564 -1.157140  0.247215   
0610009L18Rik    15.925952       -0.257931  0.221957 -1.162077  0.245204   
0610010F05Rik    56.505150       -0.011450  0.150439 -0.076108  0.939333   
...                    ...             ...       ...       ...       ...   
mt-Tv           436.194244       -0.306287  0.186328 -1.643802  0.100217   
mt-Tw            17.762007       -0.450843  0.253123 -1.781122  0.074893   
n-R5-8s1        135.173126       -0.636641  0.358781 -1.774454  0.075988   
n-R5s115          1.228580        0.161598  0.875979  0.184477  0.853639   
n-R5s193          9.230520   

/home/hailey/conda/envs/st/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
Fitting size factors...
... done in 0.01 seconds.

Fitting dispersions...
... done in 9.91 seconds.

Fitting dispersion trend curve...
... done in 1.57 seconds.

Fitting MAP dispersions...
... done in 7.85 seconds.

Fitting LFCs...
... done in 5.05 seconds.

Refitting 0 outliers.

Running Wald tests...
... done in 4.82 seconds.



Log2 fold change & Wald test p-value: Group PLX vs Control
                  baseMean  log2FoldChange     lfcSE      stat    pvalue  \
real_gene_name                                                             
0610005C13Rik     2.531028       -0.122998  0.550938 -0.223253  0.823339   
0610009B22Rik    68.801971        0.063211  0.143135  0.441618  0.658766   
0610009E02Rik     6.786776        0.276405  0.399785  0.691384  0.489324   
0610009L18Rik    14.560811       -0.014474  0.234830 -0.061637  0.950852   
0610010F05Rik    40.039993        0.083884  0.143395  0.584988  0.558556   
...                    ...             ...       ...       ...       ...   
mt-Tv           354.055206       -0.382916  0.239221 -1.600680  0.109448   
mt-Tw             8.265828       -0.705150  0.322237 -2.188300  0.028648   
n-R5-8s1        122.867050       -0.576668  0.309984 -1.860313  0.062841   
n-R5s193          8.573103       -0.758437  0.502077 -1.510599  0.130891   
n-R5s25           1.260467   

In [8]:
# ---- 字体（Helvetica 优先）----
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.unicode_minus": False,
})


def _infer_sex_code(sex_dirname):
    """从 sex_XXX 文件夹名猜测 sex 代号（F/M）。"""
    s = sex_dirname.lower()
    if "female" in s or s.endswith("_f") or s.endswith("f") or "sex_f" in s:
        return "F"
    if "male" in s or s.endswith("_m") or s.endswith("m") or "sex_m" in s:
        return "M"
    return re.sub(r"^sex_", "", sex_dirname, flags=re.IGNORECASE)


def count_deg_from_deseq2_all_csv(csv_path, padj_cutoff=0.05, lfc_cutoff=1.0):
    """从 DESeq2_all_*.csv 统计 up/down 个数（up/down 都是正数计数）。"""
    df = pd.read_csv(csv_path, index_col=0)

    if ("padj" not in df.columns) or ("log2FoldChange" not in df.columns):
        raise ValueError("Missing columns in {}: need 'padj' and 'log2FoldChange'".format(csv_path))

    df = df.dropna(subset=["padj", "log2FoldChange"])
    up = ((df["padj"] < padj_cutoff) & (df["log2FoldChange"] >= lfc_cutoff)).sum()
    down = ((df["padj"] < padj_cutoff) & (df["log2FoldChange"] <= -lfc_cutoff)).sum()
    return int(up), int(down)


def load_region_deg_counts_by_sex(out_dir="DEG_by_region_by_sex",
                                  padj_cutoff=0.05,
                                  lfc_cutoff=1.0,
                                  use_summary=True):
    """
    读取 out_dir/sex_*/ 下的 DESeq2_all_*.csv，返回长表：
    columns = [Sex, Region, Up, Down]
    """
    sex_dirs = sorted([
        d for d in os.listdir(out_dir)
        if os.path.isdir(os.path.join(out_dir, d)) and d.lower().startswith("sex_")
    ])
    if len(sex_dirs) == 0:
        raise ValueError("No sex_* folders found under: {}".format(out_dir))

    rows = []
    for sd in sex_dirs:
        sex_path = os.path.join(out_dir, sd)
        sex_code = _infer_sex_code(sd)

        summary_path = os.path.join(sex_path, "summary.csv")
        used_summary = False

        # 1) 优先用 summary.csv（有真实 region 名 + out_all 路径）
        if use_summary and os.path.exists(summary_path):
            sm = pd.read_csv(summary_path)
            sm = sm[sm["status"] == "ok"].copy()
            if ("region" in sm.columns) and ("out_all" in sm.columns) and (not sm.empty):
                for _, r in sm.iterrows():
                    region = r["region"]
                    csv_path = r["out_all"]
                    if (not isinstance(csv_path, str)) or (not os.path.exists(csv_path)):
                        continue
                    up, down = count_deg_from_deseq2_all_csv(csv_path, padj_cutoff, lfc_cutoff)
                    rows.append({"Sex": sex_code, "Region": str(region), "Up": up, "Down": down})
                used_summary = True

        # 2) fallback：直接 glob 文件名
        if not used_summary:
            files = sorted(glob.glob(os.path.join(sex_path, "DESeq2_all_*.csv")))
            if len(files) == 0:
                print("[warn] No DESeq2_all_*.csv found in {}".format(sex_path))
                continue

            for fp in files:
                base = os.path.basename(fp).replace("DESeq2_all_", "").replace(".csv", "")
                region = base
                up, down = count_deg_from_deseq2_all_csv(fp, padj_cutoff, lfc_cutoff)
                rows.append({"Sex": sex_code, "Region": str(region), "Up": up, "Down": down})

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("Loaded DEG counts table is empty. Check folder paths and CSV files.")
    return df


def plot_sex_specific_deg_regions(df,
                                  sex_top="F",
                                  sex_bottom="M",
                                  title="Sex-Specific DEG Patterns Across Brain Regions",
                                  region_order=None,
                                  x_max=None,
                                  figsize=(14, 8),
                                  savepath=None,
                                  dpi=300):
    """
    画“上♀下♂、左down右up”的镜像水平条形图。
    df columns: Sex, Region, Up, Down（Up/Down为正数计数）
    """
    d = df.copy()
    d["Sex"] = d["Sex"].astype(str)
    d["Region"] = d["Region"].astype(str)

    # region 顺序：优先用 top sex 的出现顺序，否则按 total DEG 降序
    if region_order is None:
        top = d[d["Sex"] == str(sex_top)]
        if not top.empty:
            region_order = top["Region"].tolist()
        else:
            tmp = d.groupby("Region")[["Up", "Down"]].sum()
            tmp["total"] = tmp["Up"] + tmp["Down"]
            region_order = tmp.sort_values("total", ascending=False).index.tolist()

    # 保证两性都有所有 region（缺的补 0）
    full = []
    for sx in [sex_top, sex_bottom]:
        for rg in region_order:
            sub = d[(d["Sex"] == str(sx)) & (d["Region"] == rg)]
            if sub.empty:
                full.append({"Sex": str(sx), "Region": rg, "Up": 0, "Down": 0})
            else:
                r = sub.iloc[0].to_dict()
                r["Sex"] = str(sx)
                r["Region"] = rg
                full.append(r)
    d2 = pd.DataFrame(full)

    # x 轴范围
    if x_max is None:
        x_max = int(max(d2["Up"].max(), d2["Down"].max()) * 1.25 + 5)
        x_max = max(x_max, 10)

    n = len(region_order)
    gap = 1.8
    y_top = np.arange(n)[::-1] + (n + gap)
    y_bot = np.arange(n)[::-1]
    ymap_top = dict(zip(region_order, y_top))
    ymap_bot = dict(zip(region_order, y_bot))

    fig, ax = plt.subplots(figsize=figsize)

    # 背景轻微分区
    ax.axhspan(y_top.min() - 0.6, y_top.max() + 0.6, alpha=0.12)
    ax.axhspan(y_bot.min() - 0.6, y_bot.max() + 0.6, alpha=0.06)

    def draw_block(sex_value, ymap):
        sub = d2[d2["Sex"] == str(sex_value)]
        for _, row in sub.iterrows():
            rg = row["Region"]
            y = ymap[rg]
            up = int(row["Up"])
            dn = int(row["Down"])

            ax.barh(y, up, height=0.75, label="Up-regulated" if (rg == region_order[0] and sex_value == sex_top) else None)
            ax.barh(y, -dn, height=0.75, label="Down-regulated" if (rg == region_order[0] and sex_value == sex_top) else None)

            # 数字标注
            if dn > 0:
                ax.text(-dn - x_max*0.02, y, str(dn), va="center", ha="right", fontsize=10, fontweight="bold")
            else:
                ax.text(-0.5, y, "0", va="center", ha="right", fontsize=10)

            if up > 0:
                ax.text(up + x_max*0.02, y, str(up), va="center", ha="left", fontsize=10, fontweight="bold")
            else:
                ax.text(0.5, y, "0", va="center", ha="left", fontsize=10)

    draw_block(sex_top, ymap_top)
    draw_block(sex_bottom, ymap_bot)

    ax.axhline((y_bot.max() + y_top.min()) / 2, linewidth=1.5)
    ax.axvline(0, linewidth=1.2)

    yticks = list(y_top) + list(y_bot)
    ylabels = region_order[::-1] + region_order[::-1]
    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels)

    ax.set_xlim(-x_max, x_max)
    ax.set_xlabel("Number of DEGs")
    ax.set_ylabel("Brain Region")
    ax.set_title(title, fontsize=16, fontweight="bold")

    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: str(int(abs(x)))))

    # legend
    handles, labels = ax.get_legend_handles_labels()
    uniq_h, uniq_l = [], []
    for h, l in zip(handles, labels):
        if l and (l not in uniq_l):
            uniq_h.append(h); uniq_l.append(l)
    if uniq_h:
        ax.legend(uniq_h, uniq_l, loc="upper center", bbox_to_anchor=(0.5, -0.08),
                  ncol=2, frameon=False, title="Regulation")

    ax.text(-x_max*0.78, y_top.max(), "FEMALES",
            fontsize=14, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="black", alpha=0.85))
    ax.text(-x_max*0.78, y_bot.min(), "MALES",
            fontsize=14, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="black", alpha=0.85))

    plt.tight_layout()

    if savepath:
        fig.savefig(savepath, dpi=dpi, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()


# ======================
# 一键：读取 → 画图
# ======================
out_dir = "DEG_by_region_by_sex"   # 改成你的输出目录
padj_cutoff = 0.05
lfc_cutoff = 1.0

df_counts = load_region_deg_counts_by_sex(
    out_dir=out_dir,
    padj_cutoff=padj_cutoff,
    lfc_cutoff=lfc_cutoff,
    use_summary=True,
)

# 保存统计表（可选）
df_counts.to_csv(os.path.join(out_dir, "deg_counts_padj{}_lfc{}.csv".format(padj_cutoff, lfc_cutoff)), index=False)

plot_sex_specific_deg_regions(
    df_counts,
    sex_top="F",
    sex_bottom="M",
    title="Sex-Specific DEG Patterns Across Brain Regions",
    savepath=os.path.join(out_dir, "sex_deg_regions.png"),
)

print("Saved:", os.path.join(out_dir, "sex_deg_regions.png"))

Saved: DEG_by_region_by_sex/sex_deg_regions.png
